<a href="https://colab.research.google.com/github/RyuichiSaito1/inflation-reddit-usa/blob/main/notebooks/gemini_2_0_flash_lite_performances_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install required packages
!pip install google-genai pandas scikit-learn

# Import libraries
import pandas as pd
import numpy as np
from google import genai
from google.genai.types import HttpOptions
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import re
import time

# Authenticate with Google Cloud (run this first)
from google.colab import auth
auth.authenticate_user()

inflation-1039-202511

In [ ]:
# Initialize client for Vertex AI
# Your project ID from the tuning job path
project_id = "634783204468"
location = "us-central1"
client = genai.Client(
    vertexai=True,
    project=project_id,
    location=location,
    http_options=HttpOptions(api_version="v1")
)

# Your tuning job name from the screenshot
tuning_job_name = "projects/634783204468/locations/us-central1/tuningJobs/2810095585925791744"

# Get the tuning job and the tuned model
tuning_job = client.tunings.get(name=tuning_job_name)

# Load test data
print("Loading test data...")
test_data_path = "/content/drive/MyDrive/world-inflation/data/reddit/production/test-data-200.csv"
df = pd.read_csv(test_data_path)
print(f"Loaded {len(df)} test samples")
print(f"Columns: {df.columns.tolist()}")
print(f"First few rows:")
print(df.head())

In [ ]:
# Define the prompt template
def create_prompt(reddit_post):
    return f"""You are a chief economist at the IMF. I would like you to infer the public perception of inflation from Reddit posts. Please classify each Reddit post into one of the following categories:
0: The post indicates deflation, such as the lower price of goods or services (e.g., "the prices are not bad"), affordable services (e.g., "this champagne is cheap and delicious"), sales information (e.g., "you can get it for only 10 dollars."), or a declining and buyer's market.
2: The post indicates or includes inflation, such as the higher price of goods or services (e.g., "it's not cheap"), the unreasonable cost of goods or services (e.g., "the food is overpriced and cold"), consumers struggling to afford necessities (e.g., "items are too expensive to buy"), shortage of goods of services, or mention about an asset bubble.
1: The post indicates neither deflation (0) nor inflation (2). This category also includes just questions to a community, social statements not personal experience, factual observations, references to originally expensive or cheap goods or services (e.g., "a gorgeous and costly dinner" or "an affordable Civic"), website promotion, authors' wishes, or illogical text.
Please choose a stronger stance when the text includes both 0 and 2 stances. If these stances are of the same degree, answer 1.
Reddit Post:
'{reddit_post}'
Classification (0, 1, or 2):"""

# Function to extract classification from response
def extract_classification(response_text):
    """Extract classification number from model response"""
    # Look for patterns like "Classification: 2" or just "2" at the end
    patterns = [
        r'Classification.*?([012])',
        r'Answer.*?([012])',
        r'Response.*?([012])',
        r'\b([012])\b'
    ]

    for pattern in patterns:
        matches = re.findall(pattern, response_text)
        if matches:
            return int(matches[-1])  # Take the last match

    # If no clear pattern, look for the first occurrence of 0, 1, or 2
    for char in response_text:
        if char in ['0', '1', '2']:
            return int(char)

    return None  # Unable to extract classification

# Function to evaluate model
def evaluate_model(model_endpoint, test_df, model_name="Model"):
    print(f"\n{'='*60}")
    print(f"🔹 Evaluating {model_name}")
    print(f"{'='*60}")

    predictions = []
    true_labels = []
    failed_predictions = 0

    # Assume the CSV has columns 'text' and 'label' - adjust as needed
    text_column = 'text' if 'text' in test_df.columns else test_df.columns[0]
    label_column = 'label' if 'label' in test_df.columns else test_df.columns[1]

    print(f"Using text column: '{text_column}' and label column: '{label_column}'")

    for idx, row in test_df.iterrows():
        reddit_post = row[text_column]
        true_label = row[label_column]

        try:
            # Create prompt
            prompt = create_prompt(reddit_post)

            # Generate prediction
            response = client.models.generate_content(
                model=model_endpoint,
                contents=prompt,
            )

            # Extract classification
            predicted_label = extract_classification(response.text)

            if predicted_label is not None:
                predictions.append(predicted_label)
                true_labels.append(true_label)

                if (idx + 1) % 10 == 0:
                    print(f"Processed {idx + 1}/{len(test_df)} samples...")
            else:
                failed_predictions += 1
                print(f"Failed to extract classification from response: {response.text[:100]}...")

        except Exception as e:
            failed_predictions += 1
            print(f"Error processing sample {idx + 1}: {e}")

        # Add small delay to avoid rate limiting
        time.sleep(0.1)

    # Calculate metrics
    if len(predictions) > 0:
        accuracy = accuracy_score(true_labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='weighted', zero_division=0)

        print(f"\n📊 RESULTS for {model_name}:")
        print(f"Total samples processed: {len(predictions)}/{len(test_df)}")
        print(f"Failed predictions: {failed_predictions}")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1-Score: {f1:.4f}")

        # Detailed classification report
        print(f"\n📋 Detailed Classification Report:")
        print(classification_report(true_labels, predictions, target_names=['Deflation (0)', 'Neutral (1)', 'Inflation (2)']))

        # Confusion Matrix
        print(f"\n🔢 Confusion Matrix:")
        cm = confusion_matrix(true_labels, predictions)
        print("True\\Pred    0    1    2")
        for i, row in enumerate(cm):
            print(f"    {i}    {row[0]:4d} {row[1]:4d} {row[2]:4d}")

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'predictions': predictions,
            'true_labels': true_labels,
            'failed_predictions': failed_predictions
        }
    else:
        print(f"❌ No valid predictions obtained for {model_name}")
        return None

Checkpoint 1 Epoch 3

In [ ]:
# Test with the default checkpoint (tuned model endpoint)
default_results = evaluate_model(tuning_job.tuned_model.endpoint, df, "DEFAULT Checkpoint")

print(f"\nEvaluation completed!")
print(f"{'='*80}")

checkpoint 2 Epoch 6/26

In [ ]:
# Test with the default checkpoint (tuned model endpoint)
default_results = evaluate_model(tuning_job.tuned_model.endpoint, df, "DEFAULT Checkpoint")

print(f"\nEvaluation completed!")
print(f"{'='*80}")

checkpoint 3 Epoch 9/26

In [ ]:
# Test with the default checkpoint (tuned model endpoint)
default_results = evaluate_model(tuning_job.tuned_model.endpoint, df, "DEFAULT Checkpoint")

print(f"\nEvaluation completed!")
print(f"{'='*80}")

checkpoint 4 11/26

In [ ]:
# Test with the default checkpoint (tuned model endpoint)
default_results = evaluate_model(tuning_job.tuned_model.endpoint, df, "DEFAULT Checkpoint")

print(f"\nEvaluation completed!")
print(f"{'='*80}")

checkpoint 5 Epoch 14/26

In [ ]:
# Test with the default checkpoint (tuned model endpoint)
default_results = evaluate_model(tuning_job.tuned_model.endpoint, df, "DEFAULT Checkpoint")

print(f"\nEvaluation completed!")
print(f"{'='*80}")

checkpoint 6 Epoch 17/26

In [ ]:
# Test with the default checkpoint (tuned model endpoint)
default_results = evaluate_model(tuning_job.tuned_model.endpoint, df, "DEFAULT Checkpoint")

print(f"\nEvaluation completed!")
print(f"{'='*80}")

checkpoint 7 Epoch 20/26

In [ ]:
# Test with the default checkpoint (tuned model endpoint)
default_results = evaluate_model(tuning_job.tuned_model.endpoint, df, "DEFAULT Checkpoint")

print(f"\nEvaluation completed!")
print(f"{'='*80}")

Checkpoint 8 Epoch 22/26

In [ ]:
# Test with the default checkpoint (tuned model endpoint)
default_results = evaluate_model(tuning_job.tuned_model.endpoint, df, "DEFAULT Checkpoint")

print(f"\nEvaluation completed!")
print(f"{'='*80}")

Checkpoint 9 Epoch 25

In [ ]:
# Test with the default checkpoint (tuned model endpoint)
default_results = evaluate_model(tuning_job.tuned_model.endpoint, df, "DEFAULT Checkpoint")

print(f"\nEvaluation completed!")
print(f"{'='*80}")

Checkpoint 10 Epoch 26/26

In [ ]:
# Test with the default checkpoint (tuned model endpoint)
default_results = evaluate_model(tuning_job.tuned_model.endpoint, df, "DEFAULT Checkpoint")

print(f"\nEvaluation completed!")
print(f"{'='*80}")

inflation-1039-202511-2

In [ ]:
# Initialize client for Vertex AI
# Your project ID from the tuning job path
project_id = "634783204468"
location = "us-central1"
client = genai.Client(
    vertexai=True,
    project=project_id,
    location=location,
    http_options=HttpOptions(api_version="v1")
)

# Your tuning job name from the screenshot
tuning_job_name = "projects/634783204468/locations/us-central1/tuningJobs/3691842338750988288"

# Get the tuning job and the tuned model
tuning_job = client.tunings.get(name=tuning_job_name)

# Load test data
print("Loading test data...")
test_data_path = "/content/drive/MyDrive/world-inflation/data/reddit/production/test-data-200.csv"
df = pd.read_csv(test_data_path)
print(f"Loaded {len(df)} test samples")
print(f"Columns: {df.columns.tolist()}")
print(f"First few rows:")
print(df.head())

In [ ]:
# Define the prompt template
def create_prompt(reddit_post):
    return f"""You are a chief economist at the IMF. I would like you to infer the public perception of inflation from Reddit posts. Please classify each Reddit post into one of the following categories:
0: The post indicates deflation, such as the lower price of goods or services (e.g., "the prices are not bad"), affordable services (e.g., "this champagne is cheap and delicious"), sales information (e.g., "you can get it for only 10 dollars."), or a declining and buyer's market.
2: The post indicates or includes inflation, such as the higher price of goods or services (e.g., "it's not cheap"), the unreasonable cost of goods or services (e.g., "the food is overpriced and cold"), consumers struggling to afford necessities (e.g., "items are too expensive to buy"), shortage of goods of services, or mention about an asset bubble.
1: The post indicates neither deflation (0) nor inflation (2). This category also includes just questions to a community, social statements not personal experience, factual observations, references to originally expensive or cheap goods or services (e.g., "a gorgeous and costly dinner" or "an affordable Civic"), website promotion, authors' wishes, or illogical text.
Please choose a stronger stance when the text includes both 0 and 2 stances. If these stances are of the same degree, answer 1.
Reddit Post:
'{reddit_post}'
Classification (0, 1, or 2):"""

# Function to extract classification from response
def extract_classification(response_text):
    """Extract classification number from model response"""
    # Look for patterns like "Classification: 2" or just "2" at the end
    patterns = [
        r'Classification.*?([012])',
        r'Answer.*?([012])',
        r'Response.*?([012])',
        r'\b([012])\b'
    ]

    for pattern in patterns:
        matches = re.findall(pattern, response_text)
        if matches:
            return int(matches[-1])  # Take the last match

    # If no clear pattern, look for the first occurrence of 0, 1, or 2
    for char in response_text:
        if char in ['0', '1', '2']:
            return int(char)

    return None  # Unable to extract classification

# Function to evaluate model
def evaluate_model(model_endpoint, test_df, model_name="Model"):
    print(f"\n{'='*60}")
    print(f"🔹 Evaluating {model_name}")
    print(f"{'='*60}")

    predictions = []
    true_labels = []
    failed_predictions = 0

    # Assume the CSV has columns 'text' and 'label' - adjust as needed
    text_column = 'text' if 'text' in test_df.columns else test_df.columns[0]
    label_column = 'label' if 'label' in test_df.columns else test_df.columns[1]

    print(f"Using text column: '{text_column}' and label column: '{label_column}'")

    for idx, row in test_df.iterrows():
        reddit_post = row[text_column]
        true_label = row[label_column]

        try:
            # Create prompt
            prompt = create_prompt(reddit_post)

            # Generate prediction
            response = client.models.generate_content(
                model=model_endpoint,
                contents=prompt,
            )

            # Extract classification
            predicted_label = extract_classification(response.text)

            if predicted_label is not None:
                predictions.append(predicted_label)
                true_labels.append(true_label)

                if (idx + 1) % 10 == 0:
                    print(f"Processed {idx + 1}/{len(test_df)} samples...")
            else:
                failed_predictions += 1
                print(f"Failed to extract classification from response: {response.text[:100]}...")

        except Exception as e:
            failed_predictions += 1
            print(f"Error processing sample {idx + 1}: {e}")

        # Add small delay to avoid rate limiting
        time.sleep(0.1)

    # Calculate metrics
    if len(predictions) > 0:
        accuracy = accuracy_score(true_labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='weighted', zero_division=0)

        print(f"\n📊 RESULTS for {model_name}:")
        print(f"Total samples processed: {len(predictions)}/{len(test_df)}")
        print(f"Failed predictions: {failed_predictions}")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1-Score: {f1:.4f}")

        # Detailed classification report
        print(f"\n📋 Detailed Classification Report:")
        print(classification_report(true_labels, predictions, target_names=['Deflation (0)', 'Neutral (1)', 'Inflation (2)']))

        # Confusion Matrix
        print(f"\n🔢 Confusion Matrix:")
        cm = confusion_matrix(true_labels, predictions)
        print("True\\Pred    0    1    2")
        for i, row in enumerate(cm):
            print(f"    {i}    {row[0]:4d} {row[1]:4d} {row[2]:4d}")

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'predictions': predictions,
            'true_labels': true_labels,
            'failed_predictions': failed_predictions
        }
    else:
        print(f"❌ No valid predictions obtained for {model_name}")
        return None

checkpoint 4 poch 5/5

In [ ]:
# Test with the default checkpoint (tuned model endpoint)
default_results = evaluate_model(tuning_job.tuned_model.endpoint, df, "DEFAULT Checkpoint")

print(f"\n✅ Evaluation completed!")
print(f"{'='*80}")

checkpoint 5 poch 5/5

In [ ]:
# Test with the default checkpoint (tuned model endpoint)
default_results = evaluate_model(tuning_job.tuned_model.endpoint, df, "DEFAULT Checkpoint")

print(f"\n✅ Evaluation completed!")
print(f"{'='*80}")

inflation-1039-202511-3

In [ ]:
# Initialize client for Vertex AI
# Your project ID from the tuning job path
project_id = "634783204468"
location = "us-central1"
client = genai.Client(
    vertexai=True,
    project=project_id,
    location=location,
    http_options=HttpOptions(api_version="v1")
)

# Your tuning job name from the screenshot
tuning_job_name = "projects/634783204468/locations/us-central1/tuningJobs/3873393698729361408"

# Get the tuning job and the tuned model
tuning_job = client.tunings.get(name=tuning_job_name)

# Load test data
print("Loading test data...")
test_data_path = "/content/drive/MyDrive/world-inflation/data/reddit/production/test-data-200.csv"
df = pd.read_csv(test_data_path)
print(f"Loaded {len(df)} test samples")
print(f"Columns: {df.columns.tolist()}")
print(f"First few rows:")
print(df.head())

In [ ]:
# Define the prompt template
def create_prompt(reddit_post):
    return f"""You are a chief economist at the IMF. I would like you to infer the public perception of inflation from Reddit posts. Please classify each Reddit post into one of the following categories:
0: The post indicates deflation, such as the lower price of goods or services (e.g., "the prices are not bad"), affordable services (e.g., "this champagne is cheap and delicious"), sales information (e.g., "you can get it for only 10 dollars."), or a declining and buyer's market.
2: The post indicates or includes inflation, such as the higher price of goods or services (e.g., "it's not cheap"), the unreasonable cost of goods or services (e.g., "the food is overpriced and cold"), consumers struggling to afford necessities (e.g., "items are too expensive to buy"), shortage of goods of services, or mention about an asset bubble.
1: The post indicates neither deflation (0) nor inflation (2). This category also includes just questions to a community, social statements not personal experience, factual observations, references to originally expensive or cheap goods or services (e.g., "a gorgeous and costly dinner" or "an affordable Civic"), website promotion, authors' wishes, or illogical text.
Please choose a stronger stance when the text includes both 0 and 2 stances. If these stances are of the same degree, answer 1.
Reddit Post:
'{reddit_post}'
Classification (0, 1, or 2):"""

# Function to extract classification from response
def extract_classification(response_text):
    """Extract classification number from model response"""
    # Look for patterns like "Classification: 2" or just "2" at the end
    patterns = [
        r'Classification.*?([012])',
        r'Answer.*?([012])',
        r'Response.*?([012])',
        r'\b([012])\b'
    ]

    for pattern in patterns:
        matches = re.findall(pattern, response_text)
        if matches:
            return int(matches[-1])  # Take the last match

    # If no clear pattern, look for the first occurrence of 0, 1, or 2
    for char in response_text:
        if char in ['0', '1', '2']:
            return int(char)

    return None  # Unable to extract classification

# Function to evaluate model
def evaluate_model(model_endpoint, test_df, model_name="Model"):
    print(f"\n{'='*60}")
    print(f"🔹 Evaluating {model_name}")
    print(f"{'='*60}")

    predictions = []
    true_labels = []
    failed_predictions = 0

    # Assume the CSV has columns 'text' and 'label' - adjust as needed
    text_column = 'text' if 'text' in test_df.columns else test_df.columns[0]
    label_column = 'label' if 'label' in test_df.columns else test_df.columns[1]

    print(f"Using text column: '{text_column}' and label column: '{label_column}'")

    for idx, row in test_df.iterrows():
        reddit_post = row[text_column]
        true_label = row[label_column]

        try:
            # Create prompt
            prompt = create_prompt(reddit_post)

            # Generate prediction
            response = client.models.generate_content(
                model=model_endpoint,
                contents=prompt,
            )

            # Extract classification
            predicted_label = extract_classification(response.text)

            if predicted_label is not None:
                predictions.append(predicted_label)
                true_labels.append(true_label)

                if (idx + 1) % 10 == 0:
                    print(f"Processed {idx + 1}/{len(test_df)} samples...")
            else:
                failed_predictions += 1
                print(f"Failed to extract classification from response: {response.text[:100]}...")

        except Exception as e:
            failed_predictions += 1
            print(f"Error processing sample {idx + 1}: {e}")

        # Add small delay to avoid rate limiting
        time.sleep(0.1)

    # Calculate metrics
    if len(predictions) > 0:
        accuracy = accuracy_score(true_labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='weighted', zero_division=0)

        print(f"\n📊 RESULTS for {model_name}:")
        print(f"Total samples processed: {len(predictions)}/{len(test_df)}")
        print(f"Failed predictions: {failed_predictions}")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1-Score: {f1:.4f}")

        # Detailed classification report
        print(f"\n📋 Detailed Classification Report:")
        print(classification_report(true_labels, predictions, target_names=['Deflation (0)', 'Neutral (1)', 'Inflation (2)']))

        # Confusion Matrix
        print(f"\n🔢 Confusion Matrix:")
        cm = confusion_matrix(true_labels, predictions)
        print("True\\Pred    0    1    2")
        for i, row in enumerate(cm):
            print(f"    {i}    {row[0]:4d} {row[1]:4d} {row[2]:4d}")

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'predictions': predictions,
            'true_labels': true_labels,
            'failed_predictions': failed_predictions
        }
    else:
        print(f"❌ No valid predictions obtained for {model_name}")
        return None

checkpoint 6 Epoch 8/8

In [ ]:
# Test with the default checkpoint (tuned model endpoint)
default_results = evaluate_model(tuning_job.tuned_model.endpoint, df, "DEFAULT Checkpoint")

print(f"\n✅ Evaluation completed!")
print(f"{'='*80}")

checkpoint 7  Epoch 8/8

In [ ]:
# Test with the default checkpoint (tuned model endpoint)
default_results = evaluate_model(tuning_job.tuned_model.endpoint, df, "DEFAULT Checkpoint")

print(f"\n✅ Evaluation completed!")
print(f"{'='*80}")

inflation-1039-202511-4

In [ ]:
# Initialize client for Vertex AI
# Your project ID from the tuning job path
project_id = "634783204468"
location = "us-central1"
client = genai.Client(
    vertexai=True,
    project=project_id,
    location=location,
    http_options=HttpOptions(api_version="v1")
)

# Your tuning job name from the screenshot
tuning_job_name = "projects/634783204468/locations/us-central1/tuningJobs/8465235731298648064"

# Get the tuning job and the tuned model
tuning_job = client.tunings.get(name=tuning_job_name)

# Load test data
print("Loading test data...")
test_data_path = "/content/drive/MyDrive/world-inflation/data/reddit/production/test-data-200.csv"
df = pd.read_csv(test_data_path)
print(f"Loaded {len(df)} test samples")
print(f"Columns: {df.columns.tolist()}")
print(f"First few rows:")
print(df.head())

In [ ]:
# Define the prompt template
def create_prompt(reddit_post):
    return f"""You are a chief economist at the IMF. I would like you to infer the public perception of inflation from Reddit posts. Please classify each Reddit post into one of the following categories:
0: The post indicates deflation, such as the lower price of goods or services (e.g., "the prices are not bad"), affordable services (e.g., "this champagne is cheap and delicious"), sales information (e.g., "you can get it for only 10 dollars."), or a declining and buyer's market.
2: The post indicates or includes inflation, such as the higher price of goods or services (e.g., "it's not cheap"), the unreasonable cost of goods or services (e.g., "the food is overpriced and cold"), consumers struggling to afford necessities (e.g., "items are too expensive to buy"), shortage of goods of services, or mention about an asset bubble.
1: The post indicates neither deflation (0) nor inflation (2). This category also includes just questions to a community, social statements not personal experience, factual observations, references to originally expensive or cheap goods or services (e.g., "a gorgeous and costly dinner" or "an affordable Civic"), website promotion, authors' wishes, or illogical text.
Please choose a stronger stance when the text includes both 0 and 2 stances. If these stances are of the same degree, answer 1.
Reddit Post:
'{reddit_post}'
Classification (0, 1, or 2):"""

# Function to extract classification from response
def extract_classification(response_text):
    """Extract classification number from model response"""
    # Look for patterns like "Classification: 2" or just "2" at the end
    patterns = [
        r'Classification.*?([012])',
        r'Answer.*?([012])',
        r'Response.*?([012])',
        r'\b([012])\b'
    ]

    for pattern in patterns:
        matches = re.findall(pattern, response_text)
        if matches:
            return int(matches[-1])  # Take the last match

    # If no clear pattern, look for the first occurrence of 0, 1, or 2
    for char in response_text:
        if char in ['0', '1', '2']:
            return int(char)

    return None  # Unable to extract classification

# Function to evaluate model
def evaluate_model(model_endpoint, test_df, model_name="Model"):
    print(f"\n{'='*60}")
    print(f"🔹 Evaluating {model_name}")
    print(f"{'='*60}")

    predictions = []
    true_labels = []
    failed_predictions = 0

    # Assume the CSV has columns 'text' and 'label' - adjust as needed
    text_column = 'text' if 'text' in test_df.columns else test_df.columns[0]
    label_column = 'label' if 'label' in test_df.columns else test_df.columns[1]

    print(f"Using text column: '{text_column}' and label column: '{label_column}'")

    for idx, row in test_df.iterrows():
        reddit_post = row[text_column]
        true_label = row[label_column]

        try:
            # Create prompt
            prompt = create_prompt(reddit_post)

            # Generate prediction
            response = client.models.generate_content(
                model=model_endpoint,
                contents=prompt,
            )

            # Extract classification
            predicted_label = extract_classification(response.text)

            if predicted_label is not None:
                predictions.append(predicted_label)
                true_labels.append(true_label)

                if (idx + 1) % 10 == 0:
                    print(f"Processed {idx + 1}/{len(test_df)} samples...")
            else:
                failed_predictions += 1
                print(f"Failed to extract classification from response: {response.text[:100]}...")

        except Exception as e:
            failed_predictions += 1
            print(f"Error processing sample {idx + 1}: {e}")

        # Add small delay to avoid rate limiting
        time.sleep(0.1)

    # Calculate metrics
    if len(predictions) > 0:
        accuracy = accuracy_score(true_labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='weighted', zero_division=0)

        print(f"\n📊 RESULTS for {model_name}:")
        print(f"Total samples processed: {len(predictions)}/{len(test_df)}")
        print(f"Failed predictions: {failed_predictions}")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1-Score: {f1:.4f}")

        # Detailed classification report
        print(f"\n📋 Detailed Classification Report:")
        print(classification_report(true_labels, predictions, target_names=['Deflation (0)', 'Neutral (1)', 'Inflation (2)']))

        # Confusion Matrix
        print(f"\n🔢 Confusion Matrix:")
        cm = confusion_matrix(true_labels, predictions)
        print("True\\Pred    0    1    2")
        for i, row in enumerate(cm):
            print(f"    {i}    {row[0]:4d} {row[1]:4d} {row[2]:4d}")

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'predictions': predictions,
            'true_labels': true_labels,
            'failed_predictions': failed_predictions
        }
    else:
        print(f"❌ No valid predictions obtained for {model_name}")
        return None

No Checkpoint

In [ ]:
# Test with the default checkpoint (tuned model endpoint)
default_results = evaluate_model(tuning_job.tuned_model.endpoint, df, "DEFAULT Checkpoint")

print(f"\n✅ Evaluation completed!")
print(f"{'='*80}")

In [ ]:
# Initialize client for Vertex AI
# Your project ID from the tuning job path
project_id = "634783204468"
location = "us-central1"
client = genai.Client(
    vertexai=True,
    project=project_id,
    location=location,
    http_options=HttpOptions(api_version="v1")
)

# Your tuning job name from the screenshot
tuning_job_name = "projects/634783204468/locations/us-central1/tuningJobs/5639719546332905472"

# Get the tuning job and the tuned model
tuning_job = client.tunings.get(name=tuning_job_name)

# Load test data
print("Loading test data...")
test_data_path = "/content/drive/MyDrive/world-inflation/data/reddit/production/test-data-200.csv"
df = pd.read_csv(test_data_path)
print(f"Loaded {len(df)} test samples")
print(f"Columns: {df.columns.tolist()}")
print(f"First few rows:")
print(df.head())

In [ ]:
# Define the prompt template
def create_prompt(reddit_post):
    return f"""You are a chief economist at the IMF. I would like you to infer the public perception of inflation from Reddit posts. Please classify each Reddit post into one of the following categories:
0: The post indicates deflation, such as the lower price of goods or services (e.g., "the prices are not bad"), affordable services (e.g., "this champagne is cheap and delicious"), sales information (e.g., "you can get it for only 10 dollars."), or a declining and buyer's market.
2: The post indicates or includes inflation, such as the higher price of goods or services (e.g., "it's not cheap"), the unreasonable cost of goods or services (e.g., "the food is overpriced and cold"), consumers struggling to afford necessities (e.g., "items are too expensive to buy"), shortage of goods of services, or mention about an asset bubble.
1: The post indicates neither deflation (0) nor inflation (2). This category also includes just questions to a community, social statements not personal experience, factual observations, references to originally expensive or cheap goods or services (e.g., "a gorgeous and costly dinner" or "an affordable Civic"), website promotion, authors' wishes, or illogical text.
Please choose a stronger stance when the text includes both 0 and 2 stances. If these stances are of the same degree, answer 1.
Reddit Post:
'{reddit_post}'
Classification (0, 1, or 2):"""

# Function to extract classification from response
def extract_classification(response_text):
    """Extract classification number from model response"""
    # Look for patterns like "Classification: 2" or just "2" at the end
    patterns = [
        r'Classification.*?([012])',
        r'Answer.*?([012])',
        r'Response.*?([012])',
        r'\b([012])\b'
    ]

    for pattern in patterns:
        matches = re.findall(pattern, response_text)
        if matches:
            return int(matches[-1])  # Take the last match

    # If no clear pattern, look for the first occurrence of 0, 1, or 2
    for char in response_text:
        if char in ['0', '1', '2']:
            return int(char)

    return None  # Unable to extract classification

# Function to evaluate model
def evaluate_model(model_endpoint, test_df, model_name="Model"):
    print(f"\n{'='*60}")
    print(f"🔹 Evaluating {model_name}")
    print(f"{'='*60}")

    predictions = []
    true_labels = []
    failed_predictions = 0

    # Assume the CSV has columns 'text' and 'label' - adjust as needed
    text_column = 'text' if 'text' in test_df.columns else test_df.columns[0]
    label_column = 'label' if 'label' in test_df.columns else test_df.columns[1]

    print(f"Using text column: '{text_column}' and label column: '{label_column}'")

    for idx, row in test_df.iterrows():
        reddit_post = row[text_column]
        true_label = row[label_column]

        try:
            # Create prompt
            prompt = create_prompt(reddit_post)

            # Generate prediction
            response = client.models.generate_content(
                model=model_endpoint,
                contents=prompt,
            )

            # Extract classification
            predicted_label = extract_classification(response.text)

            if predicted_label is not None:
                predictions.append(predicted_label)
                true_labels.append(true_label)

                if (idx + 1) % 10 == 0:
                    print(f"Processed {idx + 1}/{len(test_df)} samples...")
            else:
                failed_predictions += 1
                print(f"Failed to extract classification from response: {response.text[:100]}...")

        except Exception as e:
            failed_predictions += 1
            print(f"Error processing sample {idx + 1}: {e}")

        # Add small delay to avoid rate limiting
        time.sleep(0.1)

    # Calculate metrics
    if len(predictions) > 0:
        accuracy = accuracy_score(true_labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='weighted', zero_division=0)

        print(f"\n📊 RESULTS for {model_name}:")
        print(f"Total samples processed: {len(predictions)}/{len(test_df)}")
        print(f"Failed predictions: {failed_predictions}")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1-Score: {f1:.4f}")

        # Detailed classification report
        print(f"\n📋 Detailed Classification Report:")
        print(classification_report(true_labels, predictions, target_names=['Deflation (0)', 'Neutral (1)', 'Inflation (2)']))

        # Confusion Matrix
        print(f"\n🔢 Confusion Matrix:")
        cm = confusion_matrix(true_labels, predictions)
        print("True\\Pred    0    1    2")
        for i, row in enumerate(cm):
            print(f"    {i}    {row[0]:4d} {row[1]:4d} {row[2]:4d}")

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'predictions': predictions,
            'true_labels': true_labels,
            'failed_predictions': failed_predictions
        }
    else:
        print(f"❌ No valid predictions obtained for {model_name}")
        return None

Checkpoint 5 Epoch 14/26

In [ ]:
# Test with the default checkpoint (tuned model endpoint)
default_results = evaluate_model(tuning_job.tuned_model.endpoint, df, "DEFAULT Checkpoint")

print(f"\n✅ Evaluation completed!")
print(f"{'='*80}")

Checkpoint 10 Epock 26/26

In [ ]:
# Test with the default checkpoint (tuned model endpoint)
default_results = evaluate_model(tuning_job.tuned_model.endpoint, df, "DEFAULT Checkpoint")

print(f"\n✅ Evaluation completed!")
print(f"{'='*80}")

In [ ]:
# Initialize client for Vertex AI
# Your project ID from the tuning job path
project_id = "634783204468"
location = "us-central1"
client = genai.Client(
    vertexai=True,
    project=project_id,
    location=location,
    http_options=HttpOptions(api_version="v1")
)

# Your tuning job name from the screenshot
tuning_job_name = "projects/634783204468/locations/us-central1/tuningJobs/450306138206896128"

# Get the tuning job and the tuned model
tuning_job = client.tunings.get(name=tuning_job_name)

# Load test data
print("Loading test data...")
test_data_path = "/content/drive/MyDrive/world-inflation/data/reddit/production/test-data-200.csv"
df = pd.read_csv(test_data_path)
print(f"Loaded {len(df)} test samples")
print(f"Columns: {df.columns.tolist()}")
print(f"First few rows:")
print(df.head())

In [ ]:
# Define the prompt template
def create_prompt(reddit_post):
    return f"""You are a chief economist at the IMF. I would like you to infer the public perception of inflation from Reddit posts. Please classify each Reddit post into one of the following categories:
0: The post indicates deflation, such as the lower price of goods or services (e.g., "the prices are not bad"), affordable services (e.g., "this champagne is cheap and delicious"), sales information (e.g., "you can get it for only 10 dollars."), or a declining and buyer's market.
2: The post indicates or includes inflation, such as the higher price of goods or services (e.g., "it's not cheap"), the unreasonable cost of goods or services (e.g., "the food is overpriced and cold"), consumers struggling to afford necessities (e.g., "items are too expensive to buy"), shortage of goods of services, or mention about an asset bubble.
1: The post indicates neither deflation (0) nor inflation (2). This category also includes just questions to a community, social statements not personal experience, factual observations, references to originally expensive or cheap goods or services (e.g., "a gorgeous and costly dinner" or "an affordable Civic"), website promotion, authors' wishes, or illogical text.
Please choose a stronger stance when the text includes both 0 and 2 stances. If these stances are of the same degree, answer 1.
Reddit Post:
'{reddit_post}'
Classification (0, 1, or 2):"""

# Function to extract classification from response
def extract_classification(response_text):
    """Extract classification number from model response"""
    # Look for patterns like "Classification: 2" or just "2" at the end
    patterns = [
        r'Classification.*?([012])',
        r'Answer.*?([012])',
        r'Response.*?([012])',
        r'\b([012])\b'
    ]

    for pattern in patterns:
        matches = re.findall(pattern, response_text)
        if matches:
            return int(matches[-1])  # Take the last match

    # If no clear pattern, look for the first occurrence of 0, 1, or 2
    for char in response_text:
        if char in ['0', '1', '2']:
            return int(char)

    return None  # Unable to extract classification

# Function to evaluate model
def evaluate_model(model_endpoint, test_df, model_name="Model"):
    print(f"\n{'='*60}")
    print(f"🔹 Evaluating {model_name}")
    print(f"{'='*60}")

    predictions = []
    true_labels = []
    failed_predictions = 0

    # Assume the CSV has columns 'text' and 'label' - adjust as needed
    text_column = 'text' if 'text' in test_df.columns else test_df.columns[0]
    label_column = 'label' if 'label' in test_df.columns else test_df.columns[1]

    print(f"Using text column: '{text_column}' and label column: '{label_column}'")

    for idx, row in test_df.iterrows():
        reddit_post = row[text_column]
        true_label = row[label_column]

        try:
            # Create prompt
            prompt = create_prompt(reddit_post)

            # Generate prediction
            response = client.models.generate_content(
                model=model_endpoint,
                contents=prompt,
            )

            # Extract classification
            predicted_label = extract_classification(response.text)

            if predicted_label is not None:
                predictions.append(predicted_label)
                true_labels.append(true_label)

                if (idx + 1) % 10 == 0:
                    print(f"Processed {idx + 1}/{len(test_df)} samples...")
            else:
                failed_predictions += 1
                print(f"Failed to extract classification from response: {response.text[:100]}...")

        except Exception as e:
            failed_predictions += 1
            print(f"Error processing sample {idx + 1}: {e}")

        # Add small delay to avoid rate limiting
        time.sleep(0.1)

    # Calculate metrics
    if len(predictions) > 0:
        accuracy = accuracy_score(true_labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='weighted', zero_division=0)

        print(f"\n📊 RESULTS for {model_name}:")
        print(f"Total samples processed: {len(predictions)}/{len(test_df)}")
        print(f"Failed predictions: {failed_predictions}")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1-Score: {f1:.4f}")

        # Detailed classification report
        print(f"\n📋 Detailed Classification Report:")
        print(classification_report(true_labels, predictions, target_names=['Deflation (0)', 'Neutral (1)', 'Inflation (2)']))

        # Confusion Matrix
        print(f"\n🔢 Confusion Matrix:")
        cm = confusion_matrix(true_labels, predictions)
        print("True\\Pred    0    1    2")
        for i, row in enumerate(cm):
            print(f"    {i}    {row[0]:4d} {row[1]:4d} {row[2]:4d}")

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'predictions': predictions,
            'true_labels': true_labels,
            'failed_predictions': failed_predictions
        }
    else:
        print(f"❌ No valid predictions obtained for {model_name}")
        return None

Checkpoint 6 Epoch 12/18

In [ ]:
# Test with the default checkpoint (tuned model endpoint)
default_results = evaluate_model(tuning_job.tuned_model.endpoint, df, "DEFAULT Checkpoint")

print(f"\n✅ Evaluation completed!")
print(f"{'='*80}")

Checkpoint 10 Epoch 18/18

In [ ]:
# Test with the default checkpoint (tuned model endpoint)
default_results = evaluate_model(tuning_job.tuned_model.endpoint, df, "DEFAULT Checkpoint")

print(f"\n✅ Evaluation completed!")
print(f"{'='*80}")